# OpenLithoHub — Bring Your Own Model in 5 minutes

<a href="https://colab.research.google.com/github/OpenLithoHub/OpenLithoHub/blob/main/notebooks/colab_byom.ipynb" target="_blank"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

This notebook shows how to plug your own lithography model into the OpenLithoHub evaluation harness — same interface, same metrics, same numbers as the leaderboard CLI.

**You'll do:**
1. Install OpenLithoHub.
2. Subclass `LithographyModel` with a tiny torch model.
3. Score it with `compute_epe`, `compute_pvband`, and `check_mrc`.
4. Render a paper-ready figure.
5. (Optional) Format a leaderboard submission.

Run cells top-to-bottom with **Shift+Enter**.

## 1. Install

In [ ]:
!pip install --quiet 'openlithohub[jupyter,models]'

## 2. Define your model

Any class with a `predict(design) -> PredictionResult` method qualifies. Here we use a deliberately trivial "OPC" — a 2-pixel dilation followed by a 1-pixel erosion. Replace the body of `predict` with your real network.

The harness only requires the `mask` field on the returned `PredictionResult`. `contour` and `metadata` are optional.

In [ ]:
import torch

from openlithohub._utils.morphology import binary_dilation, binary_erosion
from openlithohub.models.base import LithographyModel, PredictionResult


class MyOPCModel(LithographyModel):
    NAME = "my-opc-tutorial"
    SUPPORTS_CURVILINEAR = False

    def predict(self, design: torch.Tensor, **kwargs) -> PredictionResult:
        mask = binary_erosion(binary_dilation(design, radius=2), radius=1)
        return PredictionResult(mask=mask, metadata={"method": "dilate+erode"})


model = MyOPCModel()
model.setup()
print("model:", model.name)

## 3. Run on a built-in design and score it

`generate_dummy_layout` returns a binary `torch.Tensor` that already satisfies basic min-width / min-spacing for the chosen pixel pitch — useful for smoke-testing without a dataset download.

In [ ]:
from openlithohub.benchmark.compliance.mrc import check_mrc
from openlithohub.benchmark.metrics.epe import compute_epe
from openlithohub.benchmark.metrics.pvband import compute_pvband
from openlithohub.data import generate_dummy_layout

target = generate_dummy_layout(size=256, seed=0)
result = model.predict(target)
predicted = result.mask

epe = compute_epe(predicted, target, pixel_size_nm=1.0)
pvb = compute_pvband(predicted, pixel_size_nm=1.0)
mrc = check_mrc(predicted, min_width_nm=10.0, min_spacing_nm=10.0, pixel_size_nm=1.0)

print("EPE :", epe)
print("PV  :", pvb)
print("MRC :", "PASS" if mrc.passed else f"FAIL ({mrc.violation_count} violations)")

## 4. Paper-ready figure

In [ ]:
from openlithohub.vis import plot_contours

fig = plot_contours(
    target,
    predicted,
    pixel_size_nm=1.0,
    title=f"{model.name} — EPE mean {epe['epe_mean_nm']:.2f} nm",
    style="ieee",
)
fig

## 5. (Optional) Format a leaderboard submission

When your numbers look good, package them into the `BenchmarkResult` schema and follow the [submission guide](https://github.com/OpenLithoHub/OpenLithoHub/blob/main/docs/leaderboard-submission.md) to open a PR.

In [ ]:
from openlithohub.leaderboard.schema import BenchmarkResult, MaskTopology, ProcessNode

submission = BenchmarkResult(
    model_name=model.name,
    dataset="lithobench",
    process_node=ProcessNode.N7,
    mask_topology=MaskTopology.MANHATTAN,
    epe_mean_nm=epe["epe_mean_nm"],
    epe_max_nm=epe["epe_max_nm"],
    mrc_violation_rate=mrc.violation_rate,
    paper_url=None,
    code_url=None,
    notes="Tutorial submission — replace before publishing.",
)
print(submission.model_dump_json(indent=2))

## Next steps

- Replace the `MyOPCModel.predict` body with your real network.
- Swap `generate_dummy_layout` for a `LithoBenchDataset` split (requires the `[data]` extra).
- Push the JSON above to `submissions/<your-handle>/<model>.json` and open a PR — the Auto-Leaderboard CI takes it from there.